In [58]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ECommerce-Fraud-Detection")
    .master("local[*]")
    .getOrCreate()
)
# Ẩn các thông báo log rác, chỉ hiển thị thông báo lỗi hệ thống
spark.sparkContext.setLogLevel("ERROR")

print("Khởi tạo SparkSession thành công")

Khởi tạo SparkSession thành công


In [59]:
# Đường dẫn lưu trữ tập dữ liệu trên hệ thống tệp tin phân tán HDFS
hdfs_path = "hdfs://localhost:9000/ecommerce-fraud-detection/transactions.csv"
# Đọc dữ liệu từ HDFS vào Spark DataFrame
df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True
)
dataframe = df.limit(5).toPandas()
display(dataframe.style.hide(axis="index"))

transaction_id,user_id,account_age_days,total_transactions_user,avg_amount_user,amount,country,bin_country,channel,merchant_category,promo_used,avs_match,cvv_result,three_ds_flag,transaction_time,shipping_distance_km,is_fraud
1,1,141,47,147.930000,84.750000,FR,FR,web,travel,0,1,1,1,2024-01-06 11:09:39,370.950000,0
2,1,141,47,147.930000,107.900000,FR,FR,web,travel,0,0,0,0,2024-01-10 03:13:47,149.620000,0
3,1,141,47,147.930000,92.360000,FR,FR,app,travel,1,1,1,1,2024-01-12 13:20:11,164.080000,0
4,1,141,47,147.930000,112.470000,FR,FR,web,fashion,0,1,1,1,2024-01-16 00:00:04,397.400000,0
5,1,141,47,147.930000,132.910000,FR,US,web,electronics,0,1,1,1,2024-01-17 08:27:31,935.280000,0


In [60]:
df.createOrReplaceTempView("ecommerce_transactions")
spark.catalog.listTables()

[Table(name='ecommerce_transactions', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [61]:
# 1. Kiểm tra số lượng dòng ban đầu trước khi xử lý NULL
total_before = df.count()
print(f"Tổng số dòng trước khi lọc Null: {total_before:,} dòng\n")

# 2. Loại bỏ các dòng (Null) ở các cột định danh bắt buộc và cột mục tiêu
df = df.dropna(subset=["transaction_id", "user_id", "amount", "is_fraud"])

# 3. Điền giá trị mặc định cho các trường phân loại bị trống
df = df.fillna({
    "country": "Unknown",
    "bin_country": "Unknown",
    "channel": "Unknown",
    "merchant_category": "Unknown"
})
# 4. Ghi nhận số lượng dòng sau khi xử lý khuyết thiếu
total_after = df.count()
print(f"Tổng số dòng sau khi lọc Null: {total_after:,} dòng")

print(f"Số lượng dòng đã bị xóa: {total_before - total_after:,} dòng\n")

Tổng số dòng trước khi lọc Null: 299,695 dòng

Tổng số dòng sau khi lọc Null: 299,695 dòng
Số lượng dòng đã bị xóa: 0 dòng



In [62]:
# 1. Ghi nhận số lượng dòng trước khi loại bỏ trùng lặp
total_before_dup = df.count()
print(f"Tổng số dòng trước khi lọc trùng: {total_before_dup:,}")

# 2. Lọc trùng dựa trên tổ hợp hành vi giao dịch bị lặp lại trong cùng một thời điểm
df = df.dropDuplicates(subset=["user_id", "amount", "merchant_category", "transaction_time"])

# 3. Ghi nhận số lượng dòng sau khi xử lý trùng lặp
total_after_dup = df.count()
print(f"Tổng số dòng sau khi lọc trùng: {total_after_dup:,}")
print(f"Số lượng bản ghi trùng lặp bị xóa bỏ: {total_before_dup - total_after_dup}")

Tổng số dòng trước khi lọc trùng: 299,695
Tổng số dòng sau khi lọc trùng: 299,695
Số lượng bản ghi trùng lặp bị xóa bỏ: 0


In [63]:
from pyspark.sql.functions import col
# 1. Ghi nhận số lượng dòng trước khi tiến hành lọc nhiễu logic
total_before_filter = df.count()
print(f"Tổng số dòng trước khi lọc nhiễu: {total_before_filter:,}")
# 2. Áp dụng bộ lọc đa điều kiện
df = df.filter(
    (col("amount") > 0) &
    (col("avg_amount_user") > 0) &
    (col("account_age_days") >= 0) &
    (col("total_transactions_user") >= 0) &
    (col("shipping_distance_km") >= 0) &
    (col("shipping_distance_km") <= 10000) &
    (col("amount") <= (col("avg_amount_user") * 100))
)
# 3. Ghi nhận số lượng dòng sau khi thanh lọc dữ liệu bất hợp lý
total_after_filter = df.count()
print(f"Tổng số dòng sau khi lọc nhiễu: {total_after_filter:,}")
print(f"Số lượng bản ghi nhiễu logic bị loại bỏ: {total_before_filter - total_after_filter}")

Tổng số dòng trước khi lọc nhiễu: 299,695
Tổng số dòng sau khi lọc nhiễu: 299,579
Số lượng bản ghi nhiễu logic bị loại bỏ: 116


In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType, TimestampType, BooleanType

# 1. Hiển thị lược đồ kiểu dữ liệu thô trước khi chuyển đổi
print("Lược đồ kiểu dữ liệu trước khi chuẩn hóa")
df.printSchema()

# 2. Thực hiện ép kiểu dữ liệu chuẩn hóa toàn diện
df_casted = df \
    .withColumn("amount", col("amount").cast(DoubleType())) \
    .withColumn("avg_amount_user", col("avg_amount_user").cast(DoubleType())) \
    .withColumn("shipping_distance_km", col("shipping_distance_km").cast(DoubleType())) \
    .withColumn("account_age_days", col("account_age_days").cast(IntegerType())) \
    .withColumn("total_transactions_user", col("total_transactions_user").cast(IntegerType())) \
    .withColumn("is_fraud", col("is_fraud").cast(IntegerType())) \
    .withColumn("transaction_time", col("transaction_time").cast(TimestampType())) \
    .withColumn("promo_used", col("promo_used").cast(BooleanType())) \
    .withColumn("avs_match", col("avs_match").cast(BooleanType())) \
    .withColumn("cvv_result", col("cvv_result").cast(BooleanType())) \
    .withColumn("three_ds_flag", col("three_ds_flag").cast(BooleanType()))

# 3. Hiển thị lược đồ sau khi đã chuẩn hóa để đối chiếu
print("Lược đồ kiểu dữ liệu sau khi chuẩn hóa")
df_casted.printSchema()

# Cập nhật lại biến df chính để truyền dữ liệu sạch sang các bước sau
df = df_casted

Lược đồ kiểu dữ liệu trước khi chuẩn hóa
root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- account_age_days: integer (nullable = true)
 |-- total_transactions_user: integer (nullable = true)
 |-- avg_amount_user: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- country: string (nullable = false)
 |-- bin_country: string (nullable = false)
 |-- channel: string (nullable = false)
 |-- merchant_category: string (nullable = false)
 |-- promo_used: integer (nullable = true)
 |-- avs_match: integer (nullable = true)
 |-- cvv_result: integer (nullable = true)
 |-- three_ds_flag: integer (nullable = true)
 |-- transaction_time: timestamp (nullable = true)
 |-- shipping_distance_km: double (nullable = true)
 |-- is_fraud: integer (nullable = true)

Lược đồ kiểu dữ liệu sau khi chuẩn hóa
root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- account_age_days: integer (nullable = true)
 |--